# KOLF-A：PrePR-CT 训练与测试流程

这个 Notebook 按顺序完成以下工作：

1. 读取项目配置，确定数据、图网络、模型和结果的路径。
2. 读取单细胞数据与每种细胞类型对应的共表达图。
3. 用 control 细胞计算图节点的表达均值和方差。
4. 读取预处理好的细胞图数据，并按指定扰动划分训练集和 OOD 测试集。
5. 训练模型、保存权重，并在 OOD 测试集上预测均值和标准差。

**运行前请确认：** `project_dir`、配置文件、数据集文件、图文件及 `cells_*.pkl` 都指向你本机的实际位置。此流程仍会读取整个 `h5ad`；如果文件非常大，请先确认内存足够。


## 1. 导入依赖并加载项目函数

In [ ]:
from pathlib import Path
import pickle
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import torch
from torch_geometric.loader import DataLoader
import yaml
from tqdm.auto import tqdm

# 项目内的函数和模型定义
# project_dir 会在下一节根据配置文件确定；这里先不导入项目模块。
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# 训练和推理所需函数来自项目中的 utils.py / model.py。


## 2. 读取配置并设置路径

In [ ]:
# 修改为你电脑上 Cell-Type-Specific-Graphs-main 的实际路径
project_dir = Path(r"E:\\VSCODE\\PrePR-CT\\Cell-Type-Specific-Graphs-main")
config_file = project_dir / "training" / "config_train.yaml"

with config_file.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# 数据、图、模型和结果目录均由配置文件控制
data_dir = project_dir / config.get("data_path", "Data/")
dataset_file = data_dir / config.get("dataset_h5ad", "Neurips_Data_Processed.h5ad")
results_dir = project_dir / config.get("save_path_results", "Results/")
models_dir = project_dir / config.get("save_path_models", "model_checkpoints/")
graphs_dir = project_dir / config.get("graphs_path", "graphs/") / config.get("graphs_data", "NeurIPS")
graphs_data = config.get("graphs_data", "NeurIPS")
smiles_file = data_dir / config.get("SMILES_feat", "SMILES_feat.csv")

multi_pert = config.get("multi_pert", True)
testing_cell_type = config.get("testing_cell_type", ["B cells"])
testing_drugs = config.get("testing_drugs", ["RG7090", "HYDROXYUREA"])
params = config.get("params", {
    "hidden_channels": 128,
    "weight_decay": 1e-5,
    "in_head": 1,
    "learning_rate": -3,
    "num_epochs": 100,
    "batch_size": 256,
})

# 结果目录按测试设置区分，避免不同实验的结果混在一起
if len(testing_cell_type) > 1:
    results_dir = Path(str(results_dir) + "_unseen_pert")

# 确保输出目录存在
results_dir.mkdir(parents=True, exist_ok=True)
models_dir.mkdir(parents=True, exist_ok=True)

print(f"数据集：{dataset_file}")
print(f"图目录：{graphs_dir}")
print(f"结果目录：{results_dir}")
print(f"模型目录：{models_dir}")
print(f"测试细胞类型：{testing_cell_type}")
print(f"测试扰动：{testing_drugs}")
print(f"训练参数：{params}")


## 3. 导入项目模块并选择计算设备

In [ ]:
# 将项目根目录加入 Python 模块搜索路径
import sys
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

import utils
import model as model_module
from utils import *
from model import *

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(f"使用 GPU：{torch.cuda.get_device_name(0)}")
else:
    print("使用 CPU")


## 4. 读取单细胞数据和细胞类型共表达图

In [ ]:
# 读取 AnnData；adata.obs 中需要包含 cell_type 和 condition 两列
adata = sc.read_h5ad(dataset_file)

# 组合细胞类型与处理条件，后续用于匹配处理组和对照组
adata.obs["cov_drug"] = (
    adata.obs["cell_type"].astype(str) + "_" +
    adata.obs["condition"].astype(str)
)

# 每种细胞类型读取一张预先构建好的共表达图
cell_type_network = {}
for cell_type in adata.obs["cell_type"].unique():
    graph_file = graphs_dir / f"{cell_type}_coexpr_graph.pkl"
    cell_type_network[cell_type] = torch.load(graph_file, map_location="cpu")

# 使用 control 细胞计算图中每个基因的表达均值和方差。
# 用稀疏矩阵统计，避免把整个表达矩阵转换成稠密矩阵。
ctrl_data = adata[adata.obs["condition"].astype(str) == "control"]

for cell_type, graph in cell_type_network.items():
    gene_indices = graph.pos.tolist()
    ctrl_subset = ctrl_data[
        ctrl_data.obs["cell_type"].astype(str) == str(cell_type),
        gene_indices
    ]

    X = ctrl_subset.X
    # 稀疏矩阵和稠密矩阵都可以使用；只计算列统计量，不创建完整稠密副本
    mean_expr = np.asarray(X.mean(axis=0)).ravel()
    mean_sq = np.asarray(X.multiply(X).mean(axis=0)).ravel() if hasattr(X, "multiply") else np.mean(np.asarray(X) ** 2, axis=0)
    var_expr = np.maximum(mean_sq - mean_expr ** 2, 0)

    graph.x = torch.as_tensor(
        np.column_stack((mean_expr, var_expr)),
        dtype=torch.float32
    )


## 5. 读取预处理细胞图，并划分训练集和 OOD 测试集

In [ ]:
# 这个 pickle 文件应当由项目的数据预处理步骤生成
cells_file = data_dir / f"cells_{graphs_data}.pkl"
with cells_file.open("rb") as f:
    cell_all_data = pickle.load(f)

# 找出属于指定测试细胞类型和测试扰动的组合
testing_cov_drug = set(
    adata.obs.loc[
        adata.obs["condition"].isin(testing_drugs)
        & adata.obs["cell_type"].isin(testing_cell_type),
        "cov_drug"
    ].unique()
)

# 测试组合留作 OOD 测试，其余组合用于训练
cells_ood = [cell for cell in cell_all_data if cell.cov_drug in testing_cov_drug]
cells_train = [cell for cell in cell_all_data if cell.cov_drug not in testing_cov_drug]

print(f"训练样本数：{len(cells_train)}")
print(f"OOD 测试样本数：{len(cells_ood)}")

train_dataloader = DataLoader(
    cells_train,
    batch_size=params["batch_size"],
    shuffle=True
)
ood_loader = DataLoader(
    cells_ood,
    batch_size=params["batch_size"],
    shuffle=False
)


## 6. 训练模型并保存权重

In [ ]:
# 固定随机种子，便于复现实验
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# 初始化图神经网络
gnn = GNN(
    total_genes=len(adata.var),
    num_perts=110,
    act_fct="Sigmoid",
    hidden_channels=params["hidden_channels"],
    in_head=params["in_head"],
    multi_pert=multi_pert,
).to(device)

# 使用训练集拟合模型
gnn = train(
    model=gnn,
    num_epochs=params["num_epochs"],
    lr=10 ** params["learning_rate"],
    weight_decay=params["weight_decay"],
    multi_pert=multi_pert,
    cell_type_network=cell_type_network,
    train_loader=train_dataloader,
)

# 保存训练好的模型参数；训练结束后直接使用内存中的模型即可，无需立刻重复加载
model_file = models_dir / f"{graphs_data}_model.pt"
torch.save(gnn.state_dict(), model_file)
print(f"模型权重已保存：{model_file}")


## 7. 为测试扰动筛选差异表达基因（DEGs）

In [ ]:
# 对每个测试组合，将处理组与同一细胞类型的 control 比较。
# 这里只保留 BH 校正后 p 值小于 0.05 的基因，并把基因名转成索引。
degs_dict = {}

for cov_drug in tqdm(sorted(testing_cov_drug), desc="计算测试组合的 DEGs"):
    cell_type, drug = cov_drug.split("_", 1)
    if cell_type not in cell_type_network:
        continue

    gene_indices = cell_type_network[cell_type].pos.tolist()
    control_label = f"{cell_type}_control"

    adata_cov = adata[
        adata.obs["cov_drug"].isin([cov_drug, control_label]),
        gene_indices
    ].copy()

    # 如果该组合或对应对照没有细胞，就跳过该组合
    if adata_cov.n_obs == 0 or adata_cov.obs["cov_drug"].nunique() < 2:
        continue

    sc.tl.rank_genes_groups(
        adata_cov,
        groupby="cov_drug",
        rankby_abs=True,
        method="t-test",
        corr_method="benjamini-hochberg",
        reference=control_label,
        n_genes=adata_cov.n_vars,
    )

    deg_table = sc.get.rank_genes_groups_df(adata_cov, group=cov_drug)
    significant_genes = deg_table.loc[
        deg_table["pvals_adj"] < 0.05, "names"
    ].dropna().tolist()

    # 沿用项目内 rank_genes 函数的基因排序逻辑
    ranked_genes = rank_genes(deg_table[deg_table["names"].isin(significant_genes)])
    gene_mask = adata_cov.var_names.isin(ranked_genes)
    degs_dict[cov_drug] = np.flatnonzero(gene_mask)

print(f"已生成 {len(degs_dict)} 个测试组合的 DEG 索引。")


## 8. 在 OOD 测试集上预测均值与标准差

In [ ]:
# 预测扰动后的平均表达
print("开始预测平均表达……")
Inference_multi_pert(
    cell_type_network,
    gnn,
    str(results_dir),
    ood_loader,
    adata,
    degs_dict=degs_dict,
    mean_or_std=True,
    plot=True,
    multi_pert=multi_pert,
)

# 预测扰动后表达的标准差
print("开始预测表达标准差……")
Inference_multi_pert(
    cell_type_network,
    gnn,
    str(results_dir),
    ood_loader,
    adata,
    degs_dict=degs_dict,
    mean_or_std=False,
    plot=True,
    multi_pert=multi_pert,
)
